# 01 · Llamando a un LLM, paso a paso

**Objetivo.** Ver las piezas de una llamada a un modelo: mensajes, parámetros, petición HTTP y lectura de la respuesta. Primero con un modelo local (LM Studio) y después con Gemini, para comprobar que la idea es la misma aunque cambie el proveedor.

## 0. Preparación

Las llaves viven en `.env` y la configuración en `config/`; nunca se escriben en el notebook. LM Studio debe estar abierto, con un modelo cargado y el servidor local activo.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from receipt_validation.config import load_settings

settings = load_settings(ROOT)
print(f"Project root: {ROOT}")

In [ ]:
import requests

LOCAL_URL = f"{settings['lmstudio_base_url'].rstrip('/')}/chat/completions"
LOCAL_MODEL = settings["default_lmstudio_model"]
GEMINI_MODEL = settings["default_gemini_model"]
GEMINI_URL = settings['gemini_base_url']

PROMPT = "Invent a name and a one-sentence slogan for a coffee shop on the Moon."

print("LOCAL MODEL - URL:", f"{LOCAL_MODEL} - {LOCAL_URL}")
print("GEMINI MODEL - URL:", f"{GEMINI_MODEL} - {GEMINI_URL}")

## 1. Anatomía de una llamada

Una llamada se arma por bloques. El primero es la **conversación**: una lista de mensajes con roles. `system` define el comportamiento del modelo y `user` lleva la pregunta.

In [ ]:
# TODO: Construye la lista `messages` con dos mensajes:
#   - rol "system": comportamiento del modelo (copywriter creativo, respuesta de una línea)
#   - rol "user": el PROMPT
messages = []
messages

El segundo bloque son los **parámetros**: qué modelo usar, cuánta variación permitir (`temperature`) y el máximo de tokens de salida (`max_tokens`).

In [ ]:
# TODO: Arma el `payload` con: model (LOCAL_MODEL), messages, temperature=0.7 y max_tokens=600.
payload = {}
payload

El tercer bloque es la **petición HTTP**. El servidor responde un JSON con la respuesta, el razonamiento (si el modelo razona) y el conteo de tokens.

In [ ]:
# TODO: Haz el POST a LOCAL_URL con el payload (requests.post, timeout=120)
# y decodifica el JSON de la respuesta en `data`.
data = {}
data

Último bloque: **leer la respuesta**. El texto vive en `choices[0].message.content`, el razonamiento en `reasoning_content` y los tokens en `usage`.

In [ ]:
# TODO: Extrae de `data`:
#   - answer: choices[0].message.content
#   - reasoning: choices[0].message.reasoning_content (puede no existir)
#   - usage: el diccionario de tokens
answer = ""
reasoning = ""
usage = {}

Con un helper pequeño mostramos los resultados de forma legible en vez de JSON crudo.

In [ ]:
from IPython.display import Markdown, display

def show(title, body):
    display(Markdown(f"#### {title}\n\n{body}\n\n---"))

show("Answer", answer)
if reasoning:
    show("Reasoning", reasoning)
show("Tokens", f"input: {usage.get('prompt_tokens')} · output: {usage.get('completion_tokens')}")

## 2. Una función reutilizable

Los cuatro bloques anteriores caben en una sola función. A partir de aquí, cada experimento es una línea.

In [ ]:
def call_local(prompt, temperature=0.7, max_tokens=600):
    # TODO 1: Arma el payload igual que en la sección anterior (system + user).
    # TODO 2: POST a LOCAL_URL y valida con raise_for_status().
    # TODO 3: Regresa un dict con answer, reasoning y usage.
    raise NotImplementedError("Completa la llamada local")

## 3. Experimento: temperatura

Tarea creativa (nombre + eslogan) y **dos corridas por temperatura**. Con `0.0` las dos corridas salen casi idénticas: el modelo siempre elige el token más probable. Al subir la temperatura, cada corrida produce ideas distintas. Más diversidad, no «mejor».

> **Pausa:** antes de ejecutar, predigan qué pasa al repetir la llamada con `0.0` y con `1.2`.

In [ ]:
# TODO: Corre call_local dos veces por cada temperatura (0.0, 0.7, 1.2)
# y muestra cada resultado con show(). ¿Qué cambia entre corridas?

## 4. Cómo razona el modelo

Los modelos de razonamiento «piensan» antes de responder y ese proceso llega en `reasoning_content`, separado de la respuesta final. Verlo ayuda a entender qué hace el modelo con la instrucción.

In [ ]:
# TODO: Pregunta al modelo cuántas letras 'r' tiene "strawberry"
# y muestra por separado el reasoning y la respuesta final.

## 5. Lo mismo con Gemini

Cambia la forma: el API de *interactions* recibe `input`, `system_instruction` y `generation_config`, la llave viaja en el header `x-goog-api-key`, y la respuesta llega en `steps` (el paso `model_output` trae el texto). Las piezas siguen siendo las mismas: mensajes, parámetros, petición y lectura. Por eso podemos repetir el experimento y comparar proveedores.

In [ ]:
def call_gemini(prompt, temperature=0.7, max_tokens=600):
    # TODO 1: Arma el payload del API de interactions: model, input,
    #   system_instruction y generation_config (temperature, max_output_tokens).
    # TODO 2: Manda la llave en el header "x-goog-api-key" y haz POST a GEMINI_URL.
    # TODO 3: La respuesta llega en steps: junta los item["text"] de los steps
    #   tipo "model_output" y regresa {"answer": ...}.
    raise NotImplementedError("Completa la llamada a Gemini")

In [ ]:
# TODO: Repite el experimento de temperaturas (0.0, 0.7, 1.2) x 2 corridas con call_gemini.

## 6. Lo mismo con la librería oficial (`google-genai`)

En producción rara vez armamos el REST a mano: los proveedores publican SDKs que esconden la plomería (URL, headers, parseo). La llamada es la misma de siempre — modelo, instrucción de sistema, parámetros y contenido — pero en tres líneas. La llave la toma del entorno que ya cargamos desde `.env`.

In [ ]:
from google import genai

# TODO: Crea el cliente con la llave de settings y llama client.models.generate_content
# con GEMINI_MODEL, system_instruction, temperature=0.7, max_output_tokens=600 y PROMPT.
# Muestra response.text con show().

## 7. Multimodal: transcribir un audio

Los modelos actuales no solo reciben texto. Con el mismo SDK subimos un archivo de audio a la File API y lo pasamos como contenido junto con la instrucción. Primero lo escuchamos, luego se lo damos al modelo.

In [ ]:
from IPython.display import Audio

audio_path = ROOT / "data" / "audios" / "audio_shut_down_computer.mp3"
Audio(str(audio_path))

In [ ]:
# TODO 1: Sube el audio con client.files.upload(file=audio_path).
# TODO 2: Pide al modelo "Generate a transcript of the audio." pasando [prompt, uploaded_file].
# TODO 3: Muestra la transcripción con show().
# Crédito del audio: https://pixabay.com/es/sound-effects/gente-shut-down-your-computer-ai-mee-universe-142703/

## 8. Cierre

- Toda llamada tiene las mismas piezas: mensajes, parámetros, petición y lectura.
- `temperature` controla diversidad, no calidad.
- `max_tokens` es un límite de salida, no una meta.
- Cambiar de proveedor solo cambia la forma del payload, no el experimento.

<details><summary><strong>Chequeo para revelar al final</strong></summary>

1. ¿En qué campo llega la respuesta en cada API?
2. ¿Dónde aparecen los tokens de entrada y salida?
3. ¿Qué error esperan si la llave o el servidor local no están disponibles?
</details>